# Brain Tumor MRI Classification — VGG16 (improved, Kaggle GPU)

**Before running (Kaggle):** in the right-hand panel open **Settings** and set
**Accelerator → GPU T4 x2** (or P100) and **Internet → On**. Then click **Run All**.

What's improved over the first VGG16 version:
- `vgg16.preprocess_input` (the ImageNet normalization VGG16 was trained with) instead of `/255`
- `GlobalAveragePooling2D` head instead of `Flatten` (far fewer parameters, less overfitting)
- **Two-phase training**: train the new head with the backbone frozen, then fine-tune blocks 4–5 at a low learning rate
- Label smoothing, early stopping on validation accuracy, learning-rate reduction
- Fast `tf.data` pipeline with GPU augmentation

At the end it saves **`VGG16_Brain_Tumor_v2.keras`** (optimizer state removed, so the file is smaller).

In [ ]:
import os, glob
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input
from sklearn.metrics import classification_report, confusion_matrix

print("TensorFlow:", tf.__version__, "| Keras:", keras.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPUs:", gpus)
assert gpus, "No GPU! Settings -> Accelerator -> GPU, then run again."

SEED = 42
keras.utils.set_random_seed(SEED)

In [ ]:
# ---------------- Dataset ----------------
# Works whether or not the dataset is attached with "Add Input"
import kagglehub
path = kagglehub.dataset_download("masoudnickparvar/brain-tumor-mri-dataset")
train_path = glob.glob(os.path.join(path, "**", "Training"), recursive=True)[0]
test_path = glob.glob(os.path.join(path, "**", "Testing"), recursive=True)[0]
print(train_path, os.listdir(train_path))
print(test_path, os.listdir(test_path))

In [ ]:
# ---------------- tf.data pipeline ----------------
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

common = dict(image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode="categorical",
              interpolation="bilinear")

train_ds, val_ds = keras.utils.image_dataset_from_directory(
    train_path, validation_split=0.2, subset="both", seed=SEED, shuffle=True, **common)
test_ds = keras.utils.image_dataset_from_directory(test_path, shuffle=False, **common)

class_labels = train_ds.class_names
print(class_labels)   # ['glioma', 'meningioma', 'notumor', 'pituitary']

augment = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.04),          # about +/- 15 degrees
    layers.RandomZoom(0.1),
    layers.RandomTranslation(0.05, 0.05),
    layers.RandomContrast(0.1),
], name="augment")

def prep_train(x, y):
    return preprocess_input(augment(x, training=True)), y

def prep_eval(x, y):
    return preprocess_input(x), y

train_ds = train_ds.map(prep_train, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
val_ds = val_ds.map(prep_eval, num_parallel_calls=AUTOTUNE).cache().prefetch(AUTOTUNE)
test_ds = test_ds.map(prep_eval, num_parallel_calls=AUTOTUNE).cache().prefetch(AUTOTUNE)

In [ ]:
# ---------------- Build model ----------------
base = VGG16(weights="imagenet", include_top=False, input_shape=IMG_SIZE + (3,))
base.trainable = False

inputs = keras.Input(shape=IMG_SIZE + (3,))
x = base(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.4)(x)
x = layers.Dense(256, activation="relu")(x)
x = layers.Dropout(0.4)(x)
outputs = layers.Dense(len(class_labels), activation="softmax")(x)
model = keras.Model(inputs, outputs, name="vgg16_brain_tumor")

loss = keras.losses.CategoricalCrossentropy(label_smoothing=0.1)
model.summary()

In [ ]:
# ---------------- Phase 1: train the new head (backbone frozen) ----------------
model.compile(optimizer=keras.optimizers.Adam(1e-3), loss=loss, metrics=["accuracy"])
history1 = model.fit(train_ds, validation_data=val_ds, epochs=6)

In [ ]:
# ---------------- Phase 2: fine-tune blocks 4 and 5 ----------------
base.trainable = True
fine_tune_from = [l.name for l in base.layers].index("block4_conv1")
for layer in base.layers[:fine_tune_from]:
    layer.trainable = False
print("Trainable backbone layers:", [l.name for l in base.layers if l.trainable])

model.compile(optimizer=keras.optimizers.Adam(1e-5), loss=loss, metrics=["accuracy"])

callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=6, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=3, min_lr=1e-7, verbose=1),
]
history2 = model.fit(train_ds, validation_data=val_ds, epochs=30, callbacks=callbacks)

In [ ]:
# ---------------- Training curves (both phases) ----------------
acc = history1.history["accuracy"] + history2.history["accuracy"]
val_acc = history1.history["val_accuracy"] + history2.history["val_accuracy"]
loss_h = history1.history["loss"] + history2.history["loss"]
val_loss = history1.history["val_loss"] + history2.history["val_loss"]
p1 = len(history1.history["accuracy"])

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(acc, label="Train Acc"); plt.plot(val_acc, label="Val Acc")
plt.axvline(p1 - 0.5, color="gray", ls="--", label="Fine-tuning starts")
plt.xlabel("Epoch"); plt.legend(); plt.title("VGG16 Accuracy")
plt.subplot(1, 2, 2)
plt.plot(loss_h, label="Train Loss"); plt.plot(val_loss, label="Val Loss")
plt.axvline(p1 - 0.5, color="gray", ls="--")
plt.xlabel("Epoch"); plt.legend(); plt.title("VGG16 Loss")
plt.show()

In [ ]:
# ---------------- Evaluate on the held-out test set ----------------
test_loss, test_acc = model.evaluate(test_ds)
print(f"Test Loss: {test_loss:.4f}  |  Test Accuracy: {test_acc:.4f}")

y_true = np.concatenate([np.argmax(y, axis=1) for _, y in test_ds])
y_pred = np.argmax(model.predict(test_ds), axis=1)
print(classification_report(y_true, y_pred, target_names=class_labels, digits=4))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_labels, yticklabels=class_labels)
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("Confusion Matrix - VGG16")
plt.show()

In [ ]:
# ---------------- Save for the Streamlit app ----------------
# A fresh Model object over the same layers is not compiled, so the optimizer
# state is not saved -> much smaller file.
inference_model = keras.Model(model.inputs, model.outputs, name="vgg16_brain_tumor")
inference_model.save("/kaggle/working/VGG16_Brain_Tumor_v2.keras")

# Quick check: reload and compare predictions
reloaded = keras.models.load_model("/kaggle/working/VGG16_Brain_Tumor_v2.keras", compile=False)
batch = next(iter(test_ds))[0][:8]
print("Same predictions after reload:",
      np.array_equal(model.predict(batch, verbose=0).argmax(1), reloaded.predict(batch, verbose=0).argmax(1)))
print("File size: %.0f MB" % (os.path.getsize("/kaggle/working/VGG16_Brain_Tumor_v2.keras") / 1e6))

In [ ]:
# ---------------- Download link ----------------
# Click the link below, or open the "Output" section in the right panel
# (/kaggle/working) and download VGG16_Brain_Tumor_v2.keras from there.
from IPython.display import FileLink
os.chdir("/kaggle/working")
FileLink("VGG16_Brain_Tumor_v2.keras")